In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Kaggle Notebook: Build FAISS Index of Polyvore Text Embeddings
- Loads Polyvore dataset
- Generates text prompts for all items
- Uses both T4 GPUs to encode text using OpenCLIP
- Maps item split information (train/val/test/unassigned)
- Builds and saves FAISS IndexFlatIP

In [2]:
import random
import numpy as np
import torch
import os
import json
from pathlib import Path
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
!pip install -q open_clip_torch faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


## Load Polyvore & Build Metadata

In [4]:
from datasets import load_dataset

print("📥 Loading Polyvore dataset from HuggingFace (owj0421/polyvore)...")
polyvore_ds = load_dataset('owj0421/polyvore')['data']

item_metadata = {}
for i, item in enumerate(tqdm(polyvore_ds, desc="Building metadata index")):
    item_id = item['item_id']
    url_name = item.get('url_name', '')
    category = item.get('category', '')
    
    # Priority: url_name -> category -> 'clothing'
    if url_name and isinstance(url_name, str) and len(url_name.strip()) > 2:
        prompt = url_name.strip().replace("-", " ").replace("_", " ")
    else:
        if category and isinstance(category, str) and len(category.strip()) > 1:
            prompt = category.strip()
        else:
            prompt = "clothing"

    if not prompt.endswith("."):
        prompt += "."
    prompt = prompt.lower()
    
    item_metadata[item_id] = {
        'index': i,
        'prompt': prompt,
    }

all_item_ids = list(item_metadata.keys())
print(f"✅ Loaded {len(all_item_ids)} items.")

📥 Loading Polyvore dataset from HuggingFace (owj0421/polyvore)...


README.md: 0.00B [00:00, ?B/s]

data/data-00000-of-00005.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

data/data-00001-of-00005.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

data/data-00002-of-00005.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

...

data/data-00004-of-00005.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

data/data-00005-of-00005.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

Building metadata index:   0%|          | 0/251008 [00:00<?, ?it/s]

✅ Loaded 251008 items.


## Map Item Split Information

In [5]:
print("📥 Determining item splits from owj0421/polyvore-outfits (disjoint_default)...")

splits_to_check = ['train', 'validation', 'test']
item_splits = {item_id: [] for item_id in all_item_ids}

for split in splits_to_check:
    print(f"Loading {split} split...")
    outfit_ds = load_dataset('owj0421/polyvore-outfits', 'disjoint_default', split=split)
    
    for outfit in tqdm(outfit_ds, desc=f"Processing {split} outfits", leave=False):
        for item in outfit['items']:
            item_id = item['item_id']
            if item_id in item_splits and split not in item_splits[item_id]:
                item_splits[item_id].append(split)

split_counts = {'train': 0, 'validation': 0, 'test': 0, 'unassigned': 0, 'multi': 0}
for item_id, splits in item_splits.items():
    if len(splits) == 0:
        item_splits[item_id] = ['unassigned']
        split_counts['unassigned'] += 1
    elif len(splits) == 1:
        split_counts[splits[0]] += 1
    else:
        split_counts['multi'] += 1

print("✅ Split assignment complete.")
print(f"Split counts: {split_counts}")

📥 Determining item splits from owj0421/polyvore-outfits (disjoint_default)...
Loading train split...


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

...

Loading validation split...


Processing validation outfits:   0%|          | 0/3000 [00:00<?, ?it/s]

Loading test split...


Processing test outfits:   0%|          | 0/15145 [00:00<?, ?it/s]

✅ Split assignment complete.
Split counts: {'train': 68127, 'validation': 10867, 'test': 69942, 'unassigned': 98223, 'multi': 3849}


## Encode Text on Dual GPUs

In [6]:
import open_clip
from concurrent.futures import ThreadPoolExecutor

num_gpus = min(2, torch.cuda.device_count()) if torch.cuda.is_available() else 1
print(f"🚀 Using {num_gpus} GPU(s) for text encoding")

# Load models onto separate devices
clip_models = []
tokenizers = []
for i in range(num_gpus):
    device = f"cuda:{i}" if torch.cuda.is_available() else "cpu"
    print(f"Loading CLIP model on {device}...")
    model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    model = model.to(device)
    model.eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    clip_models.append((model, device))
    tokenizers.append(tokenizer)

print("✅ Models loaded.")

🚀 Using 2 GPU(s) for text encoding
Loading CLIP model on cuda:0...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading CLIP model on cuda:1...
✅ Models loaded.


In [7]:
def process_chunk(chunk_ids, gpu_idx):
    model, device = clip_models[gpu_idx]
    tokenizer = tokenizers[gpu_idx]
    
    batch_size = 256
    embeddings = []
    
    disable_pbar = (gpu_idx > 0)
    pbar = tqdm(range(0, len(chunk_ids), batch_size), desc=f"GPU {gpu_idx}", disable=disable_pbar)
    
    for i in pbar:
        batch_ids = chunk_ids[i:i+batch_size]
        prompts = [item_metadata[item_id]['prompt'] for item_id in batch_ids]
        
        tokens = tokenizer(prompts).to(device)
        with torch.no_grad():
            emb = model.encode_text(tokens)
            # L2 Normalize
            emb = emb / emb.norm(dim=-1, keepdim=True)
            
        embeddings.append(emb.cpu().numpy().astype(np.float32))
        
    return np.vstack(embeddings)

In [8]:
chunks = np.array_split(all_item_ids, num_gpus)
chunk_lists = [chunk.tolist() for chunk in chunks]

all_embeddings_list = []

print(f"\n{'='*50}")
print("Encoding texts in parallel...")
with ThreadPoolExecutor(max_workers=num_gpus) as executor:
    futures = [executor.submit(process_chunk, chunk_lists[i], i) for i in range(num_gpus)]
    
    for f in futures:
        all_embeddings_list.append(f.result())

all_text_embeddings = np.vstack(all_embeddings_list)
print(f"✅ Encoded {all_text_embeddings.shape[0]} text prompts. Shape: {all_text_embeddings.shape}")


Encoding texts in parallel...


GPU 0:   0%|          | 0/491 [00:00<?, ?it/s]

✅ Encoded 251008 text prompts. Shape: (251008, 512)


In [9]:
# Free GPU memory
del clip_models
del tokenizers
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Save Embeddings, Build FAISS, Save Metadata

In [10]:
import faiss

OUTPUT_DIR = Path("/kaggle/working/text_faiss_data")
# if running locally outside kaggle:
if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = Path("./text_faiss_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [11]:
# Save raw matrix
np.save(OUTPUT_DIR / "text_embeddings_all.npy", all_text_embeddings)
print(f"💾 Saved text_embeddings_all.npy")


💾 Saved text_embeddings_all.npy


In [12]:
# Save item mapping
with open(OUTPUT_DIR / "text_faiss_item_ids.json", "w") as f:
    json.dump(all_item_ids, f)

In [13]:
# Save split info
with open(OUTPUT_DIR / "text_faiss_split_info.json", "w") as f:
    json.dump(item_splits, f)

# Build FAISS Index (IndexFlatIP for inner product which is cosine sim on normalized vectors)
index = faiss.IndexFlatIP(512)
index.add(all_text_embeddings)

faiss.write_index(index, str(OUTPUT_DIR / "text_faiss_index.bin"))
print(f"✅ FAISS index built and saved to {OUTPUT_DIR / 'text_faiss_index.bin'}")

✅ FAISS index built and saved to /kaggle/working/text_faiss_data/text_faiss_index.bin


## Verification

In [14]:
print("\n" + "="*50)
print("🔍 Verification")

test_index = faiss.read_index(str(OUTPUT_DIR / "text_faiss_index.bin"))
print(f"Loaded index size: {test_index.ntotal}")
assert test_index.ntotal == len(all_item_ids)

sample_query = all_text_embeddings[0:1]
D, I = test_index.search(sample_query, k=5)
print(f"\nQuerying item: {all_item_ids[0]} - '{item_metadata[all_item_ids[0]]['prompt']}'")
print(f"Top 5 nearest neighbors:")
for dist, idx in zip(D[0], I[0]):
    match_id = all_item_ids[idx]
    print(f"  - [{dist:.4f}] {match_id}: '{item_metadata[match_id]['prompt']}'")

print("\n🎉 Text FAISS Index Preparation Complete!")



🔍 Verification
Loaded index size: 251008

Querying item: 211990161 - 'neck print chiffon plus size.'
Top 5 nearest neighbors:
  - [1.0000] 211990161: 'neck print chiffon plus size.'
  - [0.9412] 211119585: 'plus size print chiffon high.'
  - [0.9128] 204918407: 'chiffon printed plus size dress.'
  - [0.9053] 212376663: 'plus size transparent neck chiffon.'
  - [0.9008] 206298565: 'plus size neck chiffon dress.'

🎉 Text FAISS Index Preparation Complete!


In [15]:
import json
from pathlib import Path

# 1. Define the directory containing the files you want to upload
# Assuming your folder is named 'text_faiss_data' in the working directory
OUTPUT_DIR = Path("/kaggle/working/text_faiss_data") 

# 2. Define the metadata for the NEW dataset
kaggle_meta = {
    "title": "Polyvore Text FAISS Embeddings", # A new, distinct title
    "id": "mabdelmoneim/polyvore-text-faiss-embeddings", # Must be a unique URL slug
    "licenses": [{"name": "CC0-1.0"}]
}

# 3. Write the metadata JSON file into the folder you are uploading
with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
    json.dump(kaggle_meta, f)

# 4. Create the new dataset on Kaggle
!kaggle datasets create -p {OUTPUT_DIR} --dir-mode zip

Starting upload for file text_faiss_index.bin
100%|█████████████████████████████████████████| 490M/490M [00:05<00:00, 102MB/s]
Upload successful: text_faiss_index.bin (490MB)
Starting upload for file text_faiss_split_info.json
100%|██████████████████████████████████████| 6.22M/6.22M [00:00<00:00, 19.1MB/s]
Upload successful: text_faiss_split_info.json (6MB)
Starting upload for file text_embeddings_all.npy
100%|█████████████████████████████████████████| 490M/490M [00:04<00:00, 103MB/s]
Upload successful: text_embeddings_all.npy (490MB)
Starting upload for file text_faiss_item_ids.json
100%|██████████████████████████████████████| 3.08M/3.08M [00:00<00:00, 10.4MB/s]
Upload successful: text_faiss_item_ids.json (3MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mabdelmoneim/polyvore-text-faiss-embeddings
